# SVM カーネル（線形 vs RBF）と k 近傍法による根拠分析

書籍 第2章「分類・回帰問題の解き方」（2-1-2 評価極性分析の SVM カーネル、2-2-2 文書分類の k 近傍法）に対応する演習ノートブックです。
[BERT 演習](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch2_bert.ipynb)・[TF-IDF ベースライン演習](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch1_tfidf_baseline.ipynb)と同じ `covid-annotation.csv` を使い、CPU のみ・数秒で以下を比較します。

1. **高次元スパースなテキスト特徴量に対する SVM カーネル比較**（線形 vs 多項式 vs RBF、p.55）
2. **TF-IDF コサイン類似度 + $k$ 近傍法（k-NN）** と、予測根拠となった類似文書の可視化（p.62）

In [1]:
!curl -fLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/covid-annotation.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100  532k    0  532k    0     0  2049k      0 --:--:-- --:--:-- --:--:-- 2057k


In [2]:
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC

df = pd.read_csv("covid-annotation.csv").fillna({"コメント": ""})
df["y"] = df["ラベル"].astype(int)

train_dev_df, test_df = train_test_split(df, test_size=0.1, stratify=df["y"], random_state=42)
train_df, dev_df = train_test_split(train_dev_df, test_size=0.1, stratify=train_dev_df["y"], random_state=42)

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2)
X_train = vec.fit_transform(train_df["文書"])
X_dev = vec.transform(dev_df["文書"])
X_test = vec.transform(test_df["文書"])

y_train = train_df["y"].to_numpy()
y_dev = dev_df["y"].to_numpy()
y_test = test_df["y"].to_numpy()

print(f"訓練データ行列の形状: {X_train.shape} (非ゼロ率 = {X_train.nnz / np.prod(X_train.shape):.4%})")

訓練データ行列の形状: (810, 8091) (非ゼロ率 = 5.0734%)


文字 1〜3-gram の TF-IDF 行列は**8,000 次元超でありながら非ゼロ率が約 5% しかない「高次元スパース」**（書籍 p.55 脚注 7）な表現になります。

---

## 1. SVM：線形カーネル vs 多項式カーネル vs RBF カーネル（2-1-2、p.55）

書籍 p.55 では、高次元スパースになりやすい自然言語処理のデータでは、非線形な **RBF カーネル** や高次の多項式カーネルよりも **線形カーネル（`liblinear` 等）** が高速で扱いやすく、重みの解釈もしやすいことが解説されています。実際に精度・過学習（訓練 vs テストの差）・処理時間を比較します。

In [3]:
models = [
    ("線形カーネル (LinearSVC, C=1.0)", LinearSVC(C=1.0, random_state=42)),
    ("多項式カーネル 2次 (SVC poly d=2, C=2.0)", SVC(kernel="poly", degree=2, C=2.0, random_state=42)),
    ("多項式カーネル 3次 (SVC poly d=3, C=2.0)", SVC(kernel="poly", degree=3, C=2.0, random_state=42)),
    ("RBF カーネル (SVC rbf, C=1.0)", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)),
    ("RBF カーネル 強適合 (SVC rbf, C=10, gamma=5.0)", SVC(kernel="rbf", C=10.0, gamma=5.0, random_state=42)),
]

svm_rows = []
for name, clf in models:
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    pred_tr = clf.predict(X_train)
    pred_dv = clf.predict(X_dev)
    pred_te = clf.predict(X_test)
    elapsed = (time.perf_counter() - t0) * 1000
    svm_rows.append({
        "手法": name,
        "訓練Acc": round(accuracy_score(y_train, pred_tr), 3),
        "開発Acc": round(accuracy_score(y_dev, pred_dv), 3),
        "テストAcc": round(accuracy_score(y_test, pred_te), 3),
        "テストF1": round(f1_score(y_test, pred_te), 3),
        "学習+推論 [ms]": round(elapsed, 1),
    })

pd.DataFrame(svm_rows)

,手法,訓練Acc,開発Acc,テストAcc,テストF1,学習+推論 [ms]
0,"線形カーネル (LinearSVC, C=1.0)",1.0,0.989,0.97,0.970,25.3
1,"多項式カーネル 2次 (SVC poly d=2, C=2.0)",1.0,0.989,0.97,0.969,3789.8
2,"多項式カーネル 3次 (SVC poly d=3, C=2.0)",1.0,0.978,0.97,0.969,3899.8
3,"RBF カーネル (SVC rbf, C=1.0)",1.0,0.989,0.97,0.969,3552.7
4,"RBF カーネル 強適合 (SVC rbf, C=10, gamma=5.0)",1.0,0.889,0.90,0.909,3819.2


- 線形カーネル（`LinearSVC`）はわずか **数十ミリ秒** で動作し、テストでも高い汎化性能（0.97 前後）を出します。
- 一方、多項式カーネルや RBF カーネルは高次元スパース行列では計算時間が **100 倍以上** 重くなり、しかも精度面で線形カーネルを上回れません（特に `gamma` を強くすると訓練精度 1.000 のままテスト性能が大きく崩れる**過学習**が起きます）。

---

## 2. $k$ 近傍法（TF-IDF + コサイン類似度）と根拠事例の提示（2-2-2、p.62）

書籍 p.62 では、小規模・簡易的な手法として **TF-IDF ベクトルとコサイン類似度による $k$ 近傍法（k-NN）** が紹介されています。
k-NN は「どの訓練事例と似ていたからこのクラスに分類したのか」という**判断根拠の事例をそのまま確認でき、エラー分析しやすい**のが大きな特長です。

In [4]:
knn_rows = []
for k in [1, 3, 5, 11, 21]:
    t0 = time.perf_counter()
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(X_train, y_train)
    pred_dv = knn.predict(X_dev)
    pred_te = knn.predict(X_test)
    elapsed = (time.perf_counter() - t0) * 1000
    knn_rows.append({
        "k": k,
        "開発Acc": round(accuracy_score(y_dev, pred_dv), 3),
        "テストAcc": round(accuracy_score(y_test, pred_te), 3),
        "テストF1": round(f1_score(y_test, pred_te), 3),
        "処理時間 [ms]": round(elapsed, 1),
    })

display(pd.DataFrame(knn_rows))

# 開発データで最良の k で詳細を確認
best_k = max(knn_rows, key=lambda r: r["開発Acc"])["k"]
knn_best = KNeighborsClassifier(n_neighbors=best_k, metric="cosine").fit(X_train, y_train)
test_preds = knn_best.predict(X_test)

def inspect_neighbors(test_pos, n_show=3):
    row = test_df.iloc[test_pos]
    dists, idxs = knn_best.kneighbors(X_test[test_pos], n_neighbors=n_show)
    print(f"[テスト文書 ID={row['文書ID']}] 正解={row['ラベル']} / k-NN予測={bool(test_preds[test_pos])} | メモ={row['コメント']}")
    print(f"  本文: {row['文書'][:90]}...")
    print("  --- 参照された類似訓練事例 ---")
    for rank, (d, tr_idx) in enumerate(zip(dists[0], idxs[0]), 1):
        tr = train_df.iloc[tr_idx]
        sim = 1.0 - d
        print(f"    {rank}位 (cos={sim:.3f}, ラベル={tr['ラベル']}, メモ={tr['コメント']}): {tr['文書'][:75]}...")
    print()

correct_pos = int(np.where(test_preds == y_test)[0][0])
wrong_pos = int(np.where(test_preds != y_test)[0][0])
inspect_neighbors(correct_pos)
inspect_neighbors(wrong_pos)

,k,開発Acc,テストAcc,テストF1,処理時間 [ms]
0,1,0.922,0.86,0.863,37.5
1,3,0.922,0.94,0.940,33.3
2,5,0.933,0.93,0.932,33.4
3,11,0.900,0.88,0.887,32.0
4,21,0.911,0.87,0.881,44.6


[テスト文書 ID=789] 正解=True / k-NN予測=True | メモ=入院に伴うコロナ感染判明
  本文: 3日前から38度9分の熱と耳の奥の痛みと寒気が出始めたため、近くの呼吸器内科を受診しました。 体調悪化のため入院して点滴を受けていましたが、院内での抗原定量検査の結果、コロナウイル...
  --- 参照された類似訓練事例 ---
    1位 (cos=0.572, ラベル=True, メモ=入院に伴うコロナ感染判明): 今月10日に強い倦怠感と関節痛で目が覚め、熱を測ると38度2分あったため予約を取って夜間救急センターへ行きました。 体調悪化のため入院して点滴を受けて...
    2位 (cos=0.548, ラベル=True, メモ=入院に伴うコロナ感染判明): 最初は単なる疲れかと思っていましたが、昨日の夜になって悪寒と節々の痛みがひどくなり、職場の近くのクリニックへ行って検査を受けました。 体調悪化のため入...
    3位 (cos=0.534, ラベル=True, メモ=入院に伴うコロナ感染判明): 私の場合は、今月に入ってからに体のだるさと筋肉痛が出てから37度5分の熱が続き、念のため市立病院の内科で検査してもらいました。 体調悪化のため入院して...

[テスト文書 ID=31] 正解=False / k-NN予測=True | メモ=入院時のコロナ検査で陰性
  本文: 体調不良により入院することになりました。 入院中には新型コロナウイルスの検査も行われ、陰性の結果が出たことを確認しました。 現在は退院して家で静養している状態です。...
  --- 参照された類似訓練事例 ---
    1位 (cos=0.397, ラベル=True, メモ=入院中にコロナ感染判明): 私は自宅で療養を続けていましたが、 最近になって体力の衰えと共に症状が再び悪化し、 病院へ入院して治療を受けることになりました。 入院中に新型コロナウ...
    2位 (cos=0.341, ラベル=True, メモ=新型コロナによる体調不良・入院): 今回の新型コロナウイルスにより体調を崩したため入院することになりました。 入院中には医師や看護師から熱心に治療を受けており、退院直前まで安静に過ごすよ...
    3位 (cos=0.339, ラベ

誤分類したテスト事例（`ID=31`：入院時の検査で陰性）についても、上位の近傍訓練事例を並べて見ることで「`入院`・`検査`・`新型コロナウイルス` といった表面的な表現の類似に引っ張られ、`陰性` と `感染判明` の違いを区別できずに似た事例が選ばれた」というエラー原因を直感的に分析できます。